# Challenge 1 - T-test

In statistics, t-test is used to test if two data samples have a significant difference between their means. There are two types of t-test:

* **Student's t-test** (a.k.a. independent or uncorrelated t-test). This type of t-test is to compare the samples of **two independent populations** (e.g. test scores of students in two different classes). `scipy` provides the [`ttest_ind`](https://docs.scipy.org/doc/scipy-0.15.1/reference/generated/scipy.stats.ttest_ind.html) method to conduct student's t-test.

* **Paired t-test** (a.k.a. dependent or correlated t-test). This type of t-test is to compare the samples of **the same population** (e.g. scores of different tests of students in the same class). `scipy` provides the [`ttest_re`](https://docs.scipy.org/doc/scipy-0.15.1/reference/generated/scipy.stats.ttest_rel.html) method to conduct paired t-test.

Both types of t-tests return a number which is called the **p-value**. If p-value is below 0.05, we can confidently declare the null-hypothesis is rejected and the difference is significant. If p-value is between 0.05 and 0.1, we may also declare the null-hypothesis is rejected but we are not highly confident. If p-value is above 0.1 we do not reject the null-hypothesis.

Read more about the t-test in [this article](https://researchbasics.education.uconn.edu/t-test/) and [this Quora](https://www.quora.com/What-is-the-difference-between-a-paired-and-unpaired-t-test). Make sure you understand when to use which type of t-test. 

In [ ]:
# Import libraries

import pandas as pd
from scipy.stats import ttest_ind, ttest_rel

#### Import dataset

In this challenge we will work on the Pokemon dataset you have used last week. The goal is to test whether different groups of pokemon (e.g. Legendary vs Normal, Generation 1 vs 2, single-type vs dual-type) have different stats (e.g. HP, Attack, Defense, etc.).

In [ ]:
# Import dataset

pokemon = pd.read_csv('Pokemon.csv')

pokemon.head()

#### First we want to define a function with which we can test the means of a feature set of two samples. 

In the next cell you'll see the annotations of the Python function that explains what this function does and its arguments and returned value. This type of annotation is called **docstring** which is a convention used among Python developers. The docstring convention allows developers to write consistent tech documentations for their codes so that others can read. It also allows some websites to automatically parse the docstrings and display user-friendly documentations.

Follow the specifications of the docstring and complete the function.

In [ ]:
def t_test_features(s1, s2, features=['HP', 'Attack', 'Defense', 'Sp. Atk', 'Sp. Def', 'Speed', 'Total']):
    """Test means of a feature set of two samples
    
    Args:
        s1 (dataframe): sample 1
        s2 (dataframe): sample 2
        features (list): an array of features to test
    
    Returns:
        dict: a dictionary of t-test scores for each feature where the feature name is the key and the p-value is the value
    """
    results = {}

    for feature in features:
        # s1 and s2 are two independent samples (e.g. Legendary vs
        # non-Legendary), so we use Student's/Welch's independent t-test.
        # equal_var=False (Welch's t-test) does not assume the two
        # groups have equal variance, which is the safer default.
        stat, p_value = ttest_ind(s1[feature], s2[feature], equal_var=False)
        results[feature] = p_value
    
    return results

#### Using the `t_test_features` function, conduct t-test for Lengendary vs non-Legendary pokemons.

*Hint: your output should look like below:*

```
{'HP': 1.0026911708035284e-13,
 'Attack': 2.520372449236646e-16,
 'Defense': 4.8269984949193316e-11,
 'Sp. Atk': 1.5514614112239812e-21,
 'Sp. Def': 2.2949327864052826e-15,
 'Speed': 1.049016311882451e-18,
 'Total': 9.357954335957446e-47}
 ```

In [ ]:
legendary = pokemon[pokemon['Legendary'] == True]
non_legendary = pokemon[pokemon['Legendary'] == False]

t_test_features(legendary, non_legendary)

#### From the test results above, what conclusion can you make? Do Legendary and non-Legendary pokemons have significantly different stats on each feature?

In [ ]:
# All 7 p-values are extremely small (well below 0.05), so we reject
# the null hypothesis for every feature. Legendary and non-Legendary
# Pokemon have significantly different stats across the board -- this
# makes sense, since Legendary Pokemon are designed to be much stronger
# overall (higher Total, Attack, Sp. Atk, etc.).

#### Next, conduct t-test for Generation 1 and Generation 2 pokemons.

In [ ]:
gen1 = pokemon[pokemon['Generation'] == 1]
gen2 = pokemon[pokemon['Generation'] == 2]

t_test_features(gen1, gen2)

#### What conclusions can you make?

In [ ]:
# All p-values are well above 0.05 except Speed (p ~ 0.002). This means
# that for HP, Attack, Defense, Sp. Atk, Sp. Def, and Total, we do NOT
# reject the null hypothesis -- Generation 1 and Generation 2 Pokemon
# have statistically similar stats on those features. Speed is the one
# exception, where there IS a significant difference between the two
# generations.

#### Compare pokemons who have single type vs those having two types.

In [ ]:
single_type = pokemon[pokemon['Type 2'].isna()]
dual_type = pokemon[pokemon['Type 2'].notna()]

t_test_features(single_type, dual_type)

#### What conclusions can you make?

In [ ]:
# HP has a p-value above 0.05, so we do not reject the null hypothesis
# for HP -- single-type and dual-type Pokemon have similar HP. Every
# other feature (Attack, Defense, Sp. Atk, Sp. Def, Speed, Total) has a
# p-value well below 0.05, so dual-type Pokemon are significantly
# different (generally stronger) on those stats than single-type ones.

#### Now, we want to compare whether there are significant differences of `Attack` vs `Defense`  and  `Sp. Atk` vs `Sp. Def` of all pokemons. Please write your code below.

*Hint: are you comparing different populations or the same population?*

In [ ]:
# Attack vs Defense and Sp. Atk vs Sp. Def are two different features
# measured on the SAME group of Pokemon (the same population), so this
# calls for a paired t-test rather than the independent t-test used above.

attack_vs_defense = ttest_rel(pokemon['Attack'], pokemon['Defense'])
sp_atk_vs_sp_def = ttest_rel(pokemon['Sp. Atk'], pokemon['Sp. Def'])

print(f"Attack vs Defense:   statistic={attack_vs_defense.statistic:.4f}, p-value={attack_vs_defense.pvalue:.4g}")
print(f"Sp. Atk vs Sp. Def:  statistic={sp_atk_vs_sp_def.statistic:.4f}, p-value={sp_atk_vs_sp_def.pvalue:.4g}")

#### What conclusions can you make?

In [ ]:
# Attack vs Defense: p-value is well below 0.05, so we reject the null
# hypothesis -- across all Pokemon, Attack and Defense are significantly
# different from each other (on average, Attack tends to be higher).
#
# Sp. Atk vs Sp. Def: p-value is well above 0.05, so we do NOT reject the
# null hypothesis -- Special Attack and Special Defense are not
# significantly different from each other on average.